# New Feature Engineering Experiment

**Goal:** Engineer genuinely new features not previously tested and evaluate on AUB-only classification models.

**New features (all leakage-free — observable at publication time):**
1. `Number of Countries/Regions` — international collaboration breadth
2. `Number of Institutions` — multi-institutional collaboration
3. `h_index_std` — spread of author expertise (not just max/mean)
4. `h_index_max × avg_venue_percentile` — star author in a top venue
5. `author_count × avg_venue_percentile` — large team in a top venue
6. `h_index_max × author_count` — top author prestige × team size
7. `is_international` — binary: multi-country collaboration
8. `is_multi_institution` — binary: more than one institution

**Baseline:** current 5,027-feature set (TF-IDF + venue percentiles + author features)

**Leakage note:** All features derived from paper metadata known at submission/publication time.
Threshold computed from train years (2015–2017) only — consistent with nb23 fix.

In [ ]:
import sys
sys.path.append('../../')

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

pd.set_option('display.max_columns', None)
FEATURE_DIR = Path('../../data/features')

## 1. Load Existing Features and Cleaned Data

In [ ]:
X_train = pd.read_pickle(FEATURE_DIR / 'X_train_temporal.pkl').fillna(0)
X_test  = pd.read_pickle(FEATURE_DIR / 'X_test_temporal.pkl').fillna(0)
y_train = pd.read_pickle(FEATURE_DIR / 'y_train_cls_temporal.pkl')
y_test  = pd.read_pickle(FEATURE_DIR / 'y_test_cls_temporal.pkl')

df = pd.read_pickle('../../data/processed/cleaned_data.pkl')

print(f"Train: {X_train.shape}  |  High-impact: {y_train.mean()*100:.1f}%")
print(f"Test:  {X_test.shape}  |  High-impact: {y_test.mean()*100:.1f}%")
print(f"Cleaned data: {df.shape}")

# Sanity check: confirm threshold was computed from train years only
train_years = [2015, 2016, 2017]
test_years  = [2018, 2019, 2020]
train_mask  = df['Year'].isin(train_years)
test_mask   = df['Year'].isin(test_years)
print(f"\nTrain papers in df: {train_mask.sum()}  |  Test papers: {test_mask.sum()}")

## 2. Engineer New Features

All features computed from columns available at publication time.
Interaction terms are built from **existing** feature columns so the same leakage
profile applies.

In [ ]:
def build_new_features(df_subset, X_existing):
    """
    Build new features for the given subset of cleaned_data.
    X_existing is the corresponding existing feature matrix (for interaction terms).
    """
    feat = pd.DataFrame(index=df_subset.index)

    # --- Collaboration breadth ---
    feat['num_countries'] = pd.to_numeric(
        df_subset['Number of Countries/Regions'], errors='coerce'
    )
    feat['num_institutions'] = pd.to_numeric(
        df_subset['Number of Institutions'], errors='coerce'
    )
    feat['is_international']    = (feat['num_countries'] > 1).astype(int)
    feat['is_multi_institution'] = (feat['num_institutions'] > 1).astype(int)

    # Impute missing with median (computed on this split only)
    for col in ['num_countries', 'num_institutions']:
        feat[col] = feat[col].fillna(feat[col].median())

    # --- H-index spread ---
    def h_index_std(h_str):
        """Parse semicolon-separated H-index string and return std dev."""
        if pd.isna(h_str):
            return np.nan
        vals = []
        for tok in str(h_str).split(';'):
            tok = tok.strip()
            if tok.isdigit():
                vals.append(int(tok))
        return float(np.std(vals)) if len(vals) > 1 else 0.0

    feat['h_index_std'] = df_subset['Authors H-index'].apply(h_index_std).fillna(0)

    # --- Cross-feature interactions (from existing feature columns) ---
    # These columns must already exist in X_existing
    required = ['h_index_max', 'avg_venue_percentile', 'author_count']
    missing  = [c for c in required if c not in X_existing.columns]
    if missing:
        print(f"WARNING: missing columns for interactions: {missing}")
        print("Skipping interaction features.")
    else:
        feat['h_max_x_venue']    = X_existing['h_index_max']   * X_existing['avg_venue_percentile']
        feat['author_x_venue']   = X_existing['author_count']  * X_existing['avg_venue_percentile']
        feat['h_max_x_authors']  = X_existing['h_index_max']   * X_existing['author_count']

    return feat


df_train = df.loc[X_train.index]
df_test  = df.loc[X_test.index]

new_train = build_new_features(df_train, X_train)
new_test  = build_new_features(df_test,  X_test)

print(f"New features: {new_train.shape[1]}")
print(new_train.dtypes)
print("\nSample (train):")
print(new_train.describe())

## 3. Build Expanded Feature Matrices

In [ ]:
X_train_exp = pd.concat([X_train, new_train], axis=1)
X_test_exp  = pd.concat([X_test,  new_test],  axis=1)

# Final NaN fill (safety)
X_train_exp = X_train_exp.fillna(0)
X_test_exp  = X_test_exp.fillna(0)

print(f"Original features : {X_train.shape[1]}")
print(f"New features added: {new_train.shape[1]}")
print(f"Total features    : {X_train_exp.shape[1]}")

## 4. Evaluate — Baseline vs. Expanded

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

MODELS = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'XGBoost':             XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                                         random_state=42, scale_pos_weight=2.5, n_jobs=-1, verbosity=0),
    'LightGBM':            LGBMClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                                          random_state=42, class_weight='balanced', n_jobs=-1, verbose=-1),
}

def run_eval(X_tr, X_te, y_tr, y_te, label):
    rows = []
    for name, clf in MODELS.items():
        cv_f1  = cross_val_score(clf, X_tr, y_tr, cv=cv, scoring='f1',      n_jobs=-1).mean()
        cv_auc = cross_val_score(clf, X_tr, y_tr, cv=cv, scoring='roc_auc', n_jobs=-1).mean()
        clf.fit(X_tr, y_tr)
        y_pred      = clf.predict(X_te)
        y_pred_prob = clf.predict_proba(X_te)[:, 1]
        rows.append({
            'Feature set': label,
            'Model':       name,
            'CV F1':       cv_f1,
            'CV AUC':      cv_auc,
            'Test F1':     f1_score(y_te, y_pred),
            'Test AUC':    roc_auc_score(y_te, y_pred_prob),
            'Precision':   precision_score(y_te, y_pred),
            'Recall':      recall_score(y_te, y_pred),
        })
        print(f"  {name:25s}  CV F1={cv_f1:.4f}  Test F1={rows[-1]['Test F1']:.4f}  AUC={rows[-1]['Test AUC']:.4f}")
    return rows

print("=" * 65)
print("BASELINE (original features)")
print("=" * 65)
baseline_rows = run_eval(X_train, X_test, y_train, y_test, 'Baseline')

print("\n" + "=" * 65)
print("EXPANDED (+ collaboration + h-index std + interactions)")
print("=" * 65)
expanded_rows = run_eval(X_train_exp, X_test_exp, y_train, y_test, 'Expanded')

results_df = pd.DataFrame(baseline_rows + expanded_rows)

## 5. Summary Table

In [ ]:
pivot = results_df.pivot_table(
    index='Model', columns='Feature set',
    values=['CV F1', 'Test F1', 'Test AUC']
).round(4)

print(pivot.to_string())

# Delta: Expanded minus Baseline
print("\n--- Delta (Expanded − Baseline) ---")
for name in MODELS:
    base = results_df[(results_df.Model == name) & (results_df['Feature set'] == 'Baseline')].iloc[0]
    exp  = results_df[(results_df.Model == name) & (results_df['Feature set'] == 'Expanded')].iloc[0]
    delta_f1  = exp['Test F1']  - base['Test F1']
    delta_auc = exp['Test AUC'] - base['Test AUC']
    print(f"  {name:25s}  ΔTest F1={delta_f1:+.4f}  ΔTest AUC={delta_auc:+.4f}")

## 6. Feature Importance for New Features

In [ ]:
# Use LightGBM importance to rank ALL features and highlight the new ones
lgbm_full = LGBMClassifier(n_estimators=200, max_depth=7, learning_rate=0.05,
                             random_state=42, class_weight='balanced', n_jobs=-1, verbose=-1)
lgbm_full.fit(X_train_exp, y_train)

imp = pd.DataFrame({
    'feature':    X_train_exp.columns,
    'importance': lgbm_full.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

new_feat_names = new_train.columns.tolist()
imp['is_new'] = imp['feature'].isin(new_feat_names)

print("Top 30 features (new ones marked with *):")
for _, row in imp.head(30).iterrows():
    marker = " ***NEW***" if row['is_new'] else ""
    print(f"  {row['feature']:40s}  {row['importance']:6.0f}{marker}")

print(f"\nNew features in top 10 : {imp.head(10)['is_new'].sum()}")
print(f"New features in top 30 : {imp.head(30)['is_new'].sum()}")
print(f"\nNew feature ranks:")
for feat in new_feat_names:
    rank = imp[imp['feature'] == feat].index[0] + 1 if feat in imp['feature'].values else 'N/A'
    print(f"  {feat:35s}  rank={rank}")

## 7. Conclusion

In [ ]:
best_base = max(r['Test F1'] for r in baseline_rows)
best_exp  = max(r['Test F1'] for r in expanded_rows)
delta     = best_exp - best_base

print("=" * 60)
print("CONCLUSION")
print("=" * 60)
print(f"Best baseline Test F1 : {best_base:.4f}")
print(f"Best expanded Test F1 : {best_exp:.4f}")
print(f"Delta                 : {delta:+.4f}")

if delta > 0.01:
    print("\nResult: MEANINGFUL IMPROVEMENT — new features help.")
    print("Recommendation: add to final feature set.")
elif delta > 0.003:
    print("\nResult: MARGINAL improvement — probably noise.")
    print("Recommendation: skip unless confirmed on multiple runs.")
else:
    print("\nResult: NO improvement — new features do not help.")
    print("Recommendation: stick with original feature set.")